## `StorageClass` — dynamic provisioning

A **StorageClass** is the recipe for "how do you make a new disk."

```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: fast-ssd
  annotations:
    storageclass.kubernetes.io/is-default-class: "true"   # optional
provisioner: ebs.csi.aws.com
parameters: { type: gp3, iops: "6000" }
reclaimPolicy: Delete
volumeBindingMode: WaitForFirstConsumer
allowVolumeExpansion: true
```

The fields that matter:

- **`provisioner`** — which CSI driver to use (`ebs.csi.aws.com`, `pd.csi.storage.gke.io`, `rancher.io/local-path` on kind).
- **`parameters`** — driver-specific (`type`, `iops` for EBS; `server`, `share` for NFS).
- **`reclaimPolicy`** — `Delete` for dynamic, `Retain` for curated (next section).
- **`volumeBindingMode`** — `Immediate` (bind at PVC creation) or **`WaitForFirstConsumer`** (wait until a pod schedules, then provision in the pod's zone). Use `WaitForFirstConsumer` for zonal storage — otherwise you risk a disk in one zone and a pod in another.
- **`allowVolumeExpansion`** — `true` if the driver supports online resize.

A PVC picks a class with `storageClassName: fast-ssd`. Omit it → the cluster **default** class is used; no default and omitted → the PVC stays **unbound**. List what you have with `kubectl get storageclass` (`kubectl get sc`); on `kind` you'll see `standard`/`local-path` marked default.

On our map this is the **StorageClass** chip — the factory that turns a PVC's request into a real PV via the CSI driver below it.